In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load Wine Dataset
wine = load_wine(as_frame=True)
df = wine.frame

# Features and Target
X = df.drop("target", axis=1)
y = df["target"]

# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Baseline Model - Decision Tree
# -----------------------------
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))

# -----------------------------
# Random Forest Model
# -----------------------------
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

# -----------------------------
# Hyperparameter Tuning
# -----------------------------
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_

print("\nBest Parameters:")
print(grid.best_params_)

best_pred = best_rf.predict(X_test)

print("Tuned Random Forest Accuracy:",
      accuracy_score(y_test, best_pred))

# -----------------------------
# Feature Importance
# -----------------------------
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance)

# Plot Feature Importance
plt.figure(figsize=(8,6))
plt.barh(importance["Feature"], importance["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Feature Importance - Wine Dataset")
plt.show()

# -----------------------------
# Bias-Variance Check
# -----------------------------
train_acc = best_rf.score(X_train, y_train)
test_acc = best_rf.score(X_test, y_test)

print("\nTraining Accuracy:", train_acc)
print("Testing Accuracy :", test_acc)

if train_acc - test_acc > 0.10:
    print("Model Status: Overfitting")
elif train_acc < 0.80 and test_acc < 0.80:
    print("Model Status: Underfitting")
else:
    print("Model Status: Good Fit")

# -----------------------------
# Final Comparison
# -----------------------------
print("\nModel Comparison")
print("------------------------------")
print("Decision Tree Accuracy      :", accuracy_score(y_test, dt_pred))
print("Random Forest Accuracy      :", accuracy_score(y_test, rf_pred))
print("Tuned Random Forest Accuracy:", accuracy_score(y_test, best_pred))

print("\nSelected Model: Tuned Random Forest")
print("Reason: It achieved the highest accuracy after hyperparameter tuning and generalizes well.")